<center>
<img src="../../img/ods_stickers.jpg" />
    
## [mlcourse.ai](https://mlcourse.ai) – دورة التعلم الآلي المفتوحة 
### <center> المؤلف: آرتشيت رونغتا
    
## <center> البرنامج التعليمي
## <center> إسناد البيانات المفقودة باستخدام Fancyimpute



مرحبا الناس!
في كثير من الأحيان، في تطبيقات العالم الحقيقي لتحليل البيانات، نواجه مشكلة البيانات المفقودة. يمكن أن يحدث هذا نتيجة لعدة أسباب مثل:
 - تم تجميع البيانات من مصادر / أوقات مختلفة 
 - تلف أثناء التخزين
 - بعض الحقول كانت اختيارية
 - الخ.
 



يحتوي هذا الدفتر على الأقسام التالية:
 1. مقدمة
 2. المشكلة
 3. احتساب KNN
 4. المقارنة والتطبيق
 5. ملخص
 6. مزيد من القراءة



في هذا البرنامج التعليمي، سنلقي نظرة على مشكلة البيانات المفقودة في تحليلات البيانات. بعد ذلك، نقوم بتصنيف الأنواع المختلفة للبيانات المفقودة ونناقش بإيجاز المشكلة المحددة التي يطرحها كل نوع محدد. أخيرًا، ننظر إلى طرق مختلفة للتعامل مع إسناد البيانات ومقارنة دقتها على مجموعة بيانات حقيقية مع الانحدار اللوجستي. نحن ننظر أيضًا إلى صحة الافتراض الشائع حول تقنيات التضمين. 



## <center> مقدمة


بشكل عام، يتم تصنيف البيانات المفقودة إلى 3 فئات. 
 - مفقود تمامًا عشوائيًا (MCAR)
 > القيم الموجودة في مجموعة بيانات مفقودة بشكل عشوائي تمامًا (MCAR) إذا كانت الأحداث التي تؤدي إلى فقدان أي عنصر بيانات معين مستقلة عن كل من المتغيرات القابلة للملاحظة والمعلمات ذات الاهتمام غير القابلة للملاحظة، وتحدث بشكل عشوائي تمامًا
 - مفقود عشوائيًا (MAR)
 >يحدث الاختفاء العشوائي (MAR) عندما لا يكون الاختفاء عشوائيًا، ولكن حيث يمكن حساب الاختفاء بشكل كامل من خلال المتغيرات حيث توجد معلومات كاملة
 - مفقود ليس عشوائيًا (MNAR)
 > المفقود ليس عشوائيًا (MNAR) (المعروف أيضًا باسم عدم الاستجابة غير القابل للتجاهل) هو البيانات التي ليست MAR أو MCAR
 
يعد تجميع البيانات من مصادر مختلفة مثالاً على MAR بينما يعد تلف البيانات مثالاً على MCAR. MNAR ليست مشكلة يمكننا حلها عن طريق الإسناد لأنها **عدم استجابة لا يمكن تجاهله.** الشيء الوحيد الذي يمكننا القيام به بشأن MNAR هو جمع المزيد من المعلومات من مصادر مختلفة أو تجاهلها تمامًا. على هذا النحو، لن نتحدث عن MNAR بعد الآن في هذا البرنامج التعليمي. 
جميع التقنيات التالية تنطبق فقط على MCAR. ومع ذلك، في سيناريوهات العالم الحقيقي، يكون MAR أكثر شيوعًا. على هذا النحو، سوف نتعامل مع MAR على أنه MCAR فقط مما يعطي تقديرًا تقريبيًا جيدًا إلى حد معقول في الممارسة العملية.



## <center> المشكلة



لنبدأ بمثال لعبة، 
\بداية{محاذاة}
\ y & = \sin(x) x\, \text{for $|x|<=6$}
\النهاية{محاذاة}


In [ ]:
import matplotlib.pyplot as plt  # plots
import numpy as np  # vectors and matrices
import pandas as pd  # tables and data manipulations
import seaborn as sns  # more plots

%matplotlib inline

In [ ]:
x = np.linspace(-6, 6)
y = np.asarray([x1 * np.sin(x1) for x1 in x])
plt.scatter(x, y)


دعونا نحذف بعض النقاط بشكل عشوائي للحصول على مجموعة بيانات MCAR


In [ ]:
missing_fraction = 0.3
indices = np.random.randint(1, len(x) - 1, size=int((1 - missing_fraction) * len(x)))
x_mcar = x[indices]
y_mcar = y[indices]

In [ ]:
plt.scatter(x_mcar, y_mcar)


خلال هذا البرنامج التعليمي، سوف نستخدم MSE كمؤشر لمدى جودة أسلوب التضمين عندما يكون لدينا مجموعة البيانات الأصلية ودقة التنبؤات عندما لا يكون لدينا


In [ ]:
from sklearn.metrics import mean_squared_error as mse


دعونا نجرب أسهل الطرق أولاً:
 - يعني
 - متوسط


In [ ]:
y_pred_mean = np.array(y)
for ind in list(set(np.linspace(0, len(x) - 1)) - set(indices)):
    y_pred_mean[int(ind)] = np.mean(y_mcar)
plt.scatter(x, y_pred_mean)
mse(y_pred_mean, y)

In [ ]:
y_pred_median = np.array(y)
for ind in list(set(np.linspace(0, len(x) - 1)) - set(indices)):
    y_pred_median[int(ind)] = np.median(y_mcar)
plt.scatter(x, y_pred_median)
mse(y_pred_median, y)

حسنا، هذا يبدو فظيعا جدا. دعونا نرى ما يقدمه موقع Fancyimpute. 
**ملاحظة: أنت بحاجة إلى TensorFlow**


In [ ]:
!pip install fancyimpute

In [ ]:
import fancyimpute

In [ ]:
y_pred_knn = np.concatenate(
    (np.array(y).reshape(-1, 1), np.array(x).reshape(-1, 1)), axis=1
)
for ind in indices:
    y_pred_knn[int(ind)] = [float("NaN"), y_pred_knn[int(ind)][1]]
y_pred_knn = fancyimpute.KNN(k=3).fit_transform(y_pred_knn)

In [ ]:
y_pred_knn_2 = [x[0] for x in y_pred_knn]

In [ ]:
plt.scatter(x, y_pred_knn_2)
mse(y_pred_knn_2, y)


كما نرى، كان أداء Fancyimpute أفضل بكثير من الطرق المتوسطة أو المتوسطة في مجموعة بيانات اللعبة هذه. 
بعد ذلك، حصلنا على فهم متعمق لكيفية عمل خوارزمية KNN الخاصة بـ Fancyimpute وتطبيقها على بعض مجموعات البيانات الحقيقية. 



## <center> إسناد KNN



>في التعرف على الأنماط، تعد خوارزمية الجيران الأقرب k طريقة غير معلمية تستخدم للتصنيف والانحدار



الافتراض وراء استخدام KNN للقيم المفقودة هو أنه يمكن تقريب قيمة النقطة من خلال قيم النقاط الأقرب إليها، بناءً على متغيرات أخرى.
تعمل خوارزمية Fancyimpute KNN عن طريق حساب أقرب جيران k الذين لديهم الميزات المفقودة المتوفرة ثم يتم وزنهم بناءً على المسافة الإقليدية من الصف المستهدف. يتم بعد ذلك حساب القيمة المفقودة كمتوسط ​​مرجح من هذه الصفوف المجاورة.
فيما يلي تطبيق لـ k = 2. لأننا نعلم أن بياناتنا مرتبة، يمكننا ترميز ذلك بشكل أكثر كفاءة. ومع ذلك، هذا ليس التنفيذ العام. نتجاهل أيضًا احتمال وجود أقرب الجيران على نفس الجانب لتقليل تعقيد الكود.


In [ ]:
y_cust = np.array(y)
for ind in indices:
    low1 = ind - 1
    while low1 in indices:
        low1 = low1 - 1
    high1 = ind + 1
    while high1 in indices:
        high1 = high1 + 1
    d1 = 1 / (ind - low1)
    d2 = 1 / (high1 - ind)
    y_cust[ind] = (d1 * y_cust[low1] + d2 * y_cust[high1]) / (d1 + d2)

In [ ]:
plt.scatter(x, y_cust)
mse(y_cust, y)


## <center> المقارنة والتطبيق



سوف نستخدم قاعدة بيانات Pima Indians Diabetes لمثال الاستخدام الخاص بنا. هذا مثال على مجموعة بيانات MAR لكننا سنتعامل معها على أنها MCAR لتحقيق أقصى استفادة مما لدينا. يمكنك تنزيل البيانات من - https://www.kaggle.com/kumargh/pimaindiansdiabetescsv


In [ ]:
df = pd.read_csv("pima-indians-diabetes.csv", header=None)

In [ ]:
df.head()

0. عدد مرات الحمل
 1. تركيز الجلوكوز في البلازما لمدة ساعتين في اختبار تحمل الجلوكوز عن طريق الفم
 2. ضغط الدم الانبساطي (مم زئبق)
 3. سماكة طيات جلد ثلاثية الرؤوس (مم)
 4. أنسولين المصل لمدة ساعتين (MU U/ml)
 5. مؤشر كتلة الجسم (الوزن بالكيلوجرام/(الطول بالمتر)^2)
 6. وظيفة نسب مرض السكري
 7. العمر (سنوات)
 8. متغير الفئة (0 أو 1)



من الواضح أن الشخص لا يمكن أن يكون سُمك طيات العضلة ثلاثية الرؤوس أقل من 0 مم. هذه قيمة مفقودة ونحتاج إلى استبدال 0 بـ NaN لإعلام خوارزمياتنا بأنها قيمة مفقودة.
من خلال قراءة الأوصاف يمكننا التأكد من أن الأعمدة 1،2،3،4،5،6 و7 لا يمكن أن تحتوي على قيم صفرية. على هذا النحو، فإننا سوف نضع علامة على 0s كمفقود. 
أيضًا، تعمل وظائف التضمين بشكل أفضل مع الميزات المقاسة، لذا سنستخدم MinMaxScaler لقياس كل ميزة بين 0 إلى 1.


In [ ]:
(df[[1, 2, 3, 4, 5, 6, 7]] == 0).sum()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

df = pd.DataFrame(
    data=MinMaxScaler().fit_transform(df.values), columns=df.columns, index=df.index
)
df[[1, 2, 3, 4, 5, 6, 7]] = df[[1, 2, 3, 4, 5, 6, 7]].replace(0, float("NaN"))

In [ ]:
df.head()


يقدم موقع Fantasyimpute العديد من الأشكال المختلفة لطرق الإسناد، ومع ذلك، فإننا نقارن فقط بين الطرق الأربعة المذكورة أدناه. يمكنك القراءة عن كل ذلك على https://pypi.org/project/fancyimpute/



الآن، سنقوم بمقارنة الانحدار اللوجستي باستخدام أربع طرق احتساب مختلفة:
 - كي إن إن
 - يعني
 - التكراري
 - سوفت إمبوت



سنقوم أولاً ببناء إطار البيانات للمراكز الثلاثة السفلية لأنه بالنسبة لـ KNN نحتاج إلى العثور على القيمة المثلى للمعلمة الفائقة. 


In [ ]:
df_mean = pd.DataFrame(
    data=fancyimpute.SimpleFill().fit_transform(df.values),
    columns=df.columns,
    index=df.index,
)
df_iterative = pd.DataFrame(
    data=fancyimpute.IterativeImputer().fit_transform(df.values),
    columns=df.columns,
    index=df.index,
)
df_soft = pd.DataFrame(
    data=fancyimpute.SoftImpute().fit_transform(df.values),
    columns=df.columns,
    index=df.index,
)

In [ ]:
from sklearn.linear_model import LogisticRegression

logisticRegr = LogisticRegression()
validation_split = 0.8
input_columns = [0, 1, 2, 3, 4, 5, 6, 7]

In [ ]:
logisticRegr.fit(
    df_mean[: int(len(df) * validation_split)][input_columns],
    df[: int(len(df) * validation_split)][8].values,
)
mean_score = logisticRegr.score(
    df_mean[int(len(df) * validation_split) :][input_columns],
    df[int(len(df) * validation_split) :][8].values,
)
mean_score

In [ ]:
logisticRegr = LogisticRegression()

logisticRegr.fit(
    df_iterative[: int(len(df) * validation_split)][input_columns],
    df[: int(len(df) * validation_split)][8].values,
)
iter_score = logisticRegr.score(
    df_iterative[int(len(df) * validation_split) :][input_columns],
    df[int(len(df) * validation_split) :][8].values,
)
iter_score

In [ ]:
logisticRegr = LogisticRegression()

logisticRegr.fit(
    df_soft[: int(len(df) * validation_split)][input_columns],
    df[: int(len(df) * validation_split)][8].values,
)
soft_score = logisticRegr.score(
    df_soft[int(len(df) * validation_split) :][input_columns],
    df[int(len(df) * validation_split) :][8].values,
)
soft_score

In [ ]:
results_knn = []

for k in range(2, 30):
    df_knn = pd.DataFrame(
        data=fancyimpute.KNN(k=k).fit_transform(df.values),
        columns=df.columns,
        index=df.index,
    )
    logisticRegr.fit(
        df_knn[: int(len(df) * validation_split)][input_columns],
        df[: int(len(df) * validation_split)][8].values,
    )
    results_knn.append(
        logisticRegr.score(
            df_knn[int(len(df) * validation_split) :][input_columns],
            df[int(len(df) * validation_split) :][8].values,
        )
    )

In [ ]:
plt.plot(results_knn)


تلخيص النتائج:
 - متوسط الاحتساب - 75.97%
 - المثبط التكراري - 77.27%
 - سوفت امبيوتر - 77.27%
 - احتساب KNN - 80.52%



غالبًا ما يُزعم أن متوسط التضمين يكون بنفس جودة الطرق الأكثر روعة مثل KNN عند استخدامها مع نماذج أكثر تعقيدًا. ولاختبارها، قمنا ببناء شبكة عصبية بسيطة وتدريبها باستخدام متوسط ​​البيانات المحسوبة ومقارنة النتائج مع بيانات KNN المحسوبة. 


In [ ]:
!pip install keras

In [ ]:
from keras.layers import Dense, Dropout
from keras.models import Sequential

model = Sequential()
model.add(Dense(10, activation="relu", input_dim=8))

model.add(Dense(10, activation="relu"))

model.add(Dense(1, activation="sigmoid"))

model.compile(loss="binary_crossentropy", optimizer="rmsprop", metrics=["accuracy"])
model.fit(
    df_mean[input_columns], df[8], batch_size=32, epochs=400, validation_split=0.2
)

In [ ]:
df_knn = pd.DataFrame(
    data=fancyimpute.KNN(k=8).fit_transform(df.values),
    columns=df.columns,
    index=df.index,
)
model = Sequential()
model.add(Dense(10, activation="relu", input_dim=8))

model.add(Dense(10, activation="relu"))

model.add(Dense(1, activation="sigmoid"))

model.compile(loss="binary_crossentropy", optimizer="rmsprop", metrics=["accuracy"])
model.fit(df_knn[input_columns], df[8], batch_size=32, epochs=400, validation_split=0.2)

In [ ]:
model.summary()

وكما يتضح من هذا الاختبار غير العلمي إلى حد كبير، فإن الحكمة الشائعة التي تعني أن الإحالة جيدة بالقدر نفسه ليست بالضرورة صحيحة. حتى مع هذه المبالغة في استخدام النموذج، فإن أداء البيانات المحسوبة لـ KNN أفضل بكثير من متوسط البيانات المحسوبة (0.8701 - العصر 396 مقابل 0.7987 - العصر 324 في هذا التشغيل)



## <center> ملخص



يتم تصنيف البيانات المفقودة على نطاق واسع إلى ثلاث فئات: MCAR وMAR وMNAR. نعرض الأداء السيئ لمتوسط ​​الإسناد والإسناد المتوسط ​​بمثال لعبة. بعد ذلك، نقوم بإنشاء فهم بديهي لتضمين KNN وكتابة نموذج التعليمات البرمجية لتنفيذه. 
وأخيرًا، قمنا بتطبيق التقنيات على مجموعة بيما الهندية لمرض السكري واستخدمنا أربع استراتيجيات احتساب مختلفة. لقد أظهرنا تفوق تقنية التضمين KNN على استراتيجيات التضمين الأخرى لكل من الانحدار اللوجستي والشبكات العصبية، مما أدى إلى تشويه الاعتقاد الشائع حول تقنيات التضمين.



## <center> مزيد من القراءة



 - https://pypi.org/project/fancyimpute/
 - https://github.com/iskandr/fancyimpute/tree/master/fancyimpute
 - https://github.com/iskandr/knnimpute/blob/master/knnimpute/few_observed_entries.py
 - https://www.ncbi.nlm.nih.gov/pmc/articles/PMC4959387/